In [ ]:
# ==================================================================================================
# FINAL H=10 ZERO-SHOT GENERALIZATION DIAGNOSTIC
#
# Purpose:
#   Diagnose why the fixed Fusion-LSTM generalizes differently from
#   Exp. 1-11 development data to independent Exp. 12-14 data.
#
# Analyses:
#   1. Development vs external feature distributions
#   2. Standardized mean shift
#   3. External samples outside development min-max range
#   4. External samples outside development mean ± 3 SD
#   5. Per-experiment feature shift
#   6. Vision vs sensor feature shift
#   7. Derivative-feature shift
#   8. Prediction confidence for correct vs incorrect cases
#   9. Per-class confidence / error behavior
#  10. Temporal transition analysis
#  11. Error analysis by experiment
#  12. Development vs external target-class distribution
#  13. Manuscript-ready diagnostic tables
#  14. Publication-ready diagnostic plots
#
# IMPORTANT:
#   This script performs DIAGNOSTIC ANALYSIS ONLY.
#   It does NOT retrain, rescale, tune, or modify predictions.
# ==================================================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

PROJECT_DIR = Path.cwd()

# ----------------------------------------------------------------------------------
# Development dataset: Exp. 1-11
#
# If your local filename differs, change ONLY this path.
# ----------------------------------------------------------------------------------

DEV_CSV = (
    PROJECT_DIR
    / "outputs"
    / "master_fusion_dataset_clean.csv"
)

# If your actual file is called master_fusion_dataset_clean(3).csv, use:
#
# DEV_CSV = Path(
#     "outputs/master_fusion_dataset_clean(3).csv"
# )


# ----------------------------------------------------------------------------------
# Final corrected zero-shot matched dataset generated in the previous step
# ----------------------------------------------------------------------------------

EXTERNAL_MATCHED_CSV = (
    PROJECT_DIR
    / "external_test"
    / "FINAL_H10_ZERO_SHOT_COCO_EVALUATION_CORRECTED"
    / "FINAL_H10_ZERO_SHOT_MATCHED_896.csv"
)


OUTPUT_DIR = (
    PROJECT_DIR
    / "external_test"
    / "FINAL_H10_ZERO_SHOT_GENERALIZATION_DIAGNOSTIC"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==================================================================================================
# 2. FINAL 20-FEATURE REPRESENTATION
# ==================================================================================================

BASE_FEATURES = [
    # Process-monitoring features
    "sensor_force",
    "sensor_rpm",
    "sensor_torque",
    "sensor_temp",

    # Vision-derived features
    "weld_width_mm",
    "weld_area_mm2",
    "Burrs_area_mm2",
    "flash_burr_area_mm2",
    "surface_groove_void_area_mm2",
    "total_defect_area_mm2",
]


DERIVATIVE_FEATURES = [
    f"{feature}_diff"
    for feature in BASE_FEATURES
]


FINAL_FEATURES = (
    BASE_FEATURES
    +
    DERIVATIVE_FEATURES
)


SENSOR_BASE = [
    "sensor_force",
    "sensor_rpm",
    "sensor_torque",
    "sensor_temp",
]

VISION_BASE = [
    "weld_width_mm",
    "weld_area_mm2",
    "Burrs_area_mm2",
    "flash_burr_area_mm2",
    "surface_groove_void_area_mm2",
    "total_defect_area_mm2",
]

SENSOR_DIFF = [
    f"{x}_diff"
    for x in SENSOR_BASE
]

VISION_DIFF = [
    f"{x}_diff"
    for x in VISION_BASE
]


CLASS_ORDER = [
    "Good",
    "Burr",
    "Flash-burr",
    "Surface-groove/void",
]


# ==================================================================================================
# 3. HELPER FUNCTIONS
# ==================================================================================================

def normalize_exp(value):

    text = str(value).strip().lower()

    if text.startswith("exp_"):
        return text

    if text.startswith("exp"):
        number = text.replace("exp", "").replace("_", "")
        return f"exp_{number}"

    return text


def normalize_class(value):

    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    mapping = {
        "good": "Good",

        "burr": "Burr",
        "burrs": "Burr",

        "flash_burr": "Flash-burr",
        "flash-bur": "Flash-burr",
        "flash_bur": "Flash-burr",
        "flash-burr": "Flash-burr",

        "surface_groove_void": "Surface-groove/void",
        "surface-groove/void": "Surface-groove/void",
        "surface_groove/void": "Surface-groove/void",
    }

    return mapping.get(
        text,
        str(value),
    )


def safe_zshift(
    dev_mean,
    ext_mean,
    dev_sd,
):
    """
    Standardized mean difference relative to development SD.
    """

    if (
        pd.isna(dev_sd)
        or dev_sd == 0
    ):
        return np.nan

    return (
        ext_mean
        -
        dev_mean
    ) / dev_sd


def magnitude_category(value):

    if pd.isna(value):
        return "Undefined"

    abs_value = abs(value)

    if abs_value < 0.2:
        return "Very small"

    if abs_value < 0.5:
        return "Small"

    if abs_value < 0.8:
        return "Moderate"

    if abs_value < 1.2:
        return "Large"

    return "Very large"


def add_derivatives_if_missing(
    df,
    experiment_column,
    frame_column,
):
    """
    Reconstruct only missing derivative columns.

    First-order differences are calculated strictly within
    each experiment after sorting by frame number.
    """

    df = df.copy()

    df = df.sort_values(
        [
            experiment_column,
            frame_column,
        ]
    ).reset_index(
        drop=True
    )

    for feature in BASE_FEATURES:

        diff_col = (
            f"{feature}_diff"
        )

        if diff_col not in df.columns:

            print(
                f"[RECONSTRUCTING] {diff_col}"
            )

            df[
                diff_col
            ] = (
                df
                .groupby(
                    experiment_column
                )[feature]
                .diff()
                .fillna(0.0)
            )

    return df


def descriptive_statistics(
    df,
    features,
):

    rows = []

    for feature in features:

        values = pd.to_numeric(
            df[feature],
            errors="coerce",
        ).dropna()

        rows.append(
            {
                "feature":
                    feature,

                "n":
                    len(values),

                "mean":
                    values.mean(),

                "sd":
                    values.std(
                        ddof=1
                    ),

                "median":
                    values.median(),

                "q1":
                    values.quantile(
                        0.25
                    ),

                "q3":
                    values.quantile(
                        0.75
                    ),

                "min":
                    values.min(),

                "max":
                    values.max(),
            }
        )

    return pd.DataFrame(
        rows
    )


# ==================================================================================================
# 4. VERIFY FILES
# ==================================================================================================

print("=" * 120)
print("FINAL H=10 ZERO-SHOT GENERALIZATION DIAGNOSTIC")
print("=" * 120)

print("\nDevelopment dataset:")
print(DEV_CSV)

if not DEV_CSV.exists():

    raise FileNotFoundError(
        "\nDevelopment CSV not found.\n"
        "Update DEV_CSV at the top of the script.\n"
        f"Current path:\n{DEV_CSV}"
    )


print("\nExternal matched zero-shot dataset:")
print(EXTERNAL_MATCHED_CSV)

if not EXTERNAL_MATCHED_CSV.exists():

    raise FileNotFoundError(
        "\nCorrected zero-shot matched CSV not found:\n"
        f"{EXTERNAL_MATCHED_CSV}"
    )


# ==================================================================================================
# 5. LOAD DATA
# ==================================================================================================

dev = pd.read_csv(
    DEV_CSV
)

ext = pd.read_csv(
    EXTERNAL_MATCHED_CSV
)


print("\nDevelopment shape:")
print(dev.shape)

print("\nExternal shape:")
print(ext.shape)


# ==================================================================================================
# 6. DETECT IDENTIFIER COLUMNS
# ==================================================================================================

if "exp_id" in dev.columns:

    DEV_EXP = "exp_id"

elif "experiment_id" in dev.columns:

    DEV_EXP = "experiment_id"

else:

    raise ValueError(
        "No experiment identifier found in development CSV."
    )


if "frame_idx" not in dev.columns:

    raise ValueError(
        "'frame_idx' missing from development CSV."
    )


DEV_FRAME = "frame_idx"


if "exp_id" not in ext.columns:

    raise ValueError(
        "'exp_id' missing from external matched CSV."
    )


if "frame_idx" in ext.columns:

    EXT_FRAME = "frame_idx"

elif "frame_idx_prediction" in ext.columns:

    EXT_FRAME = "frame_idx_prediction"

else:

    raise ValueError(
        "Could not identify external current-frame column."
    )


dev[
    DEV_EXP
] = dev[
    DEV_EXP
].apply(
    normalize_exp
)


ext[
    "exp_id"
] = ext[
    "exp_id"
].apply(
    normalize_exp
)


# ==================================================================================================
# 7. FEATURE AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("FINAL 20-FEATURE AUDIT")
print("=" * 120)


for feature in BASE_FEATURES:

    if feature not in dev.columns:

        raise ValueError(
            f"Development dataset missing base feature: {feature}"
        )

    if feature not in ext.columns:

        raise ValueError(
            f"External dataset missing base feature: {feature}"
        )


# ==================================================================================================
# 8. RECONSTRUCT MISSING DERIVATIVES IF NECESSARY
# ==================================================================================================

dev = add_derivatives_if_missing(
    dev,
    experiment_column=DEV_EXP,
    frame_column=DEV_FRAME,
)


ext = add_derivatives_if_missing(
    ext,
    experiment_column="exp_id",
    frame_column=EXT_FRAME,
)


missing_dev = [
    x
    for x in FINAL_FEATURES
    if x not in dev.columns
]

missing_ext = [
    x
    for x in FINAL_FEATURES
    if x not in ext.columns
]


if missing_dev:

    raise ValueError(
        f"Development still missing features:\n{missing_dev}"
    )


if missing_ext:

    raise ValueError(
        f"External still missing features:\n{missing_ext}"
    )


print("\nAll 20 diagnostic features available.")

print("\nFeatures:")
for feature in FINAL_FEATURES:

    print(
        " ",
        feature
    )


# ==================================================================================================
# 9. DEVELOPMENT / EXTERNAL DESCRIPTIVE STATISTICS
# ==================================================================================================

print("\n" + "=" * 120)
print("DEVELOPMENT VS EXTERNAL FEATURE DISTRIBUTIONS")
print("=" * 120)


dev_stats = descriptive_statistics(
    dev,
    FINAL_FEATURES,
)

ext_stats = descriptive_statistics(
    ext,
    FINAL_FEATURES,
)


dev_stats = dev_stats.add_prefix(
    "development_"
)

ext_stats = ext_stats.add_prefix(
    "external_"
)


comparison = pd.concat(
    [
        dev_stats,
        ext_stats.drop(
            columns=[
                "external_feature"
            ]
        ),
    ],
    axis=1,
)


comparison = comparison.rename(
    columns={
        "development_feature":
            "feature"
    }
)


# ==================================================================================================
# 10. STANDARDIZED DISTRIBUTION SHIFT
# ==================================================================================================

comparison[
    "standardized_mean_shift"
] = comparison.apply(
    lambda row:
        safe_zshift(
            row[
                "development_mean"
            ],
            row[
                "external_mean"
            ],
            row[
                "development_sd"
            ],
        ),
    axis=1,
)


comparison[
    "absolute_standardized_shift"
] = comparison[
    "standardized_mean_shift"
].abs()


comparison[
    "shift_magnitude"
] = comparison[
    "standardized_mean_shift"
].apply(
    magnitude_category
)


# ==================================================================================================
# 11. OUT-OF-DEVELOPMENT-RANGE ANALYSIS
# ==================================================================================================

outside_range_results = []


for feature in FINAL_FEATURES:

    dev_values = pd.to_numeric(
        dev[feature],
        errors="coerce",
    ).dropna()


    ext_values = pd.to_numeric(
        ext[feature],
        errors="coerce",
    ).dropna()


    dev_min = dev_values.min()
    dev_max = dev_values.max()

    dev_mean = dev_values.mean()
    dev_sd = dev_values.std(
        ddof=1
    )


    outside_minmax = (
        (
            ext_values
            <
            dev_min
        )
        |
        (
            ext_values
            >
            dev_max
        )
    )


    if (
        pd.notna(dev_sd)
        and dev_sd > 0
    ):

        outside_3sd = (
            (
                ext_values
                <
                dev_mean
                -
                3 * dev_sd
            )
            |
            (
                ext_values
                >
                dev_mean
                +
                3 * dev_sd
            )
        )

    else:

        outside_3sd = pd.Series(
            False,
            index=ext_values.index,
        )


    outside_range_results.append(
        {
            "feature":
                feature,

            "development_min":
                dev_min,

            "development_max":
                dev_max,

            "external_min":
                ext_values.min(),

            "external_max":
                ext_values.max(),

            "external_outside_dev_minmax_percent":
                100
                * outside_minmax.mean(),

            "external_outside_dev_3sd_percent":
                100
                * outside_3sd.mean(),
        }
    )


range_df = pd.DataFrame(
    outside_range_results
)


comparison = comparison.merge(
    range_df,
    on="feature",
    how="left",
)


comparison = comparison.sort_values(
    "absolute_standardized_shift",
    ascending=False,
)


comparison.to_csv(
    OUTPUT_DIR
    / "development_vs_external_feature_shift.csv",
    index=False,
)


print(
    comparison[
        [
            "feature",
            "development_mean",
            "external_mean",
            "standardized_mean_shift",
            "shift_magnitude",
            "external_outside_dev_minmax_percent",
            "external_outside_dev_3sd_percent",
        ]
    ]
    .round(3)
    .to_string(
        index=False
    )
)


# ==================================================================================================
# 12. PER-EXPERIMENT FEATURE SHIFT
# ==================================================================================================

print("\n" + "=" * 120)
print("PER-EXPERIMENT FEATURE DISTRIBUTION SHIFT")
print("=" * 120)


per_exp_shift_rows = []


for exp_id in [
    "exp_12",
    "exp_13",
    "exp_14",
]:

    exp_df = ext[
        ext[
            "exp_id"
        ]
        == exp_id
    ]


    for feature in FINAL_FEATURES:

        dev_mean = pd.to_numeric(
            dev[feature],
            errors="coerce",
        ).mean()

        dev_sd = pd.to_numeric(
            dev[feature],
            errors="coerce",
        ).std(
            ddof=1
        )

        ext_mean = pd.to_numeric(
            exp_df[feature],
            errors="coerce",
        ).mean()


        shift = safe_zshift(
            dev_mean,
            ext_mean,
            dev_sd,
        )


        per_exp_shift_rows.append(
            {
                "experiment":
                    exp_id,

                "feature":
                    feature,

                "development_mean":
                    dev_mean,

                "experiment_mean":
                    ext_mean,

                "standardized_mean_shift":
                    shift,

                "absolute_shift":
                    abs(shift)
                    if pd.notna(shift)
                    else np.nan,

                "shift_magnitude":
                    magnitude_category(
                        shift
                    ),
            }
        )


per_exp_shift_df = pd.DataFrame(
    per_exp_shift_rows
)


per_exp_shift_df.to_csv(
    OUTPUT_DIR
    / "per_experiment_feature_shift.csv",
    index=False,
)


for exp_id in [
    "exp_12",
    "exp_13",
    "exp_14",
]:

    print(
        f"\n{exp_id.upper()} — largest feature shifts"
    )

    temp = (
        per_exp_shift_df[
            per_exp_shift_df[
                "experiment"
            ]
            == exp_id
        ]
        .sort_values(
            "absolute_shift",
            ascending=False,
        )
        .head(10)
    )

    print(
        temp[
            [
                "feature",
                "standardized_mean_shift",
                "shift_magnitude",
            ]
        ]
        .round(3)
        .to_string(
            index=False
        )
    )


# ==================================================================================================
# 13. MODALITY-LEVEL SHIFT SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("MODALITY-LEVEL SHIFT SUMMARY")
print("=" * 120)


feature_groups = {
    "Sensor base":
        SENSOR_BASE,

    "Vision base":
        VISION_BASE,

    "Sensor derivatives":
        SENSOR_DIFF,

    "Vision derivatives":
        VISION_DIFF,
}


group_rows = []


for group_name, feature_list in feature_groups.items():

    temp = comparison[
        comparison[
            "feature"
        ].isin(
            feature_list
        )
    ]


    group_rows.append(
        {
            "feature_group":
                group_name,

            "mean_absolute_standardized_shift":
                temp[
                    "absolute_standardized_shift"
                ].mean(),

            "max_absolute_standardized_shift":
                temp[
                    "absolute_standardized_shift"
                ].max(),

            "mean_external_outside_minmax_percent":
                temp[
                    "external_outside_dev_minmax_percent"
                ].mean(),
        }
    )


group_df = pd.DataFrame(
    group_rows
)


print(
    group_df
    .round(3)
    .to_string(
        index=False
    )
)


group_df.to_csv(
    OUTPUT_DIR
    / "feature_group_shift_summary.csv",
    index=False,
)


# ==================================================================================================
# 14. EXTERNAL PREDICTION / GT AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("EXTERNAL PREDICTION ERROR AUDIT")
print("=" * 120)


required_columns = [
    "true_class",
    "pred_class",
    "max_probability",
]


for column in required_columns:

    if column not in ext.columns:

        print(
            f"[WARNING] {column} not present."
        )


ext[
    "true_class"
] = ext[
    "true_class"
].apply(
    normalize_class
)


ext[
    "pred_class"
] = ext[
    "pred_class"
].apply(
    normalize_class
)


ext[
    "correct_prediction"
] = (
    ext[
        "true_class"
    ]
    ==
    ext[
        "pred_class"
    ]
)


print("\nCorrect / incorrect:")

print(
    ext[
        "correct_prediction"
    ]
    .value_counts()
)


# ==================================================================================================
# 15. CONFIDENCE ANALYSIS
# ==================================================================================================

if "max_probability" in ext.columns:

    print("\n" + "=" * 120)
    print("PREDICTION CONFIDENCE ANALYSIS")
    print("=" * 120)


    confidence_summary = (
        ext
        .groupby(
            "correct_prediction"
        )[
            "max_probability"
        ]
        .agg(
            [
                "count",
                "mean",
                "std",
                "median",
                "min",
                "max",
            ]
        )
    )


    print(
        confidence_summary
        .round(4)
    )


    confidence_summary.to_csv(
        OUTPUT_DIR
        / "confidence_correct_vs_incorrect.csv"
    )


    class_confidence = (
        ext
        .groupby(
            [
                "true_class",
                "correct_prediction",
            ]
        )[
            "max_probability"
        ]
        .agg(
            [
                "count",
                "mean",
                "median",
            ]
        )
        .reset_index()
    )


    class_confidence.to_csv(
        OUTPUT_DIR
        / "confidence_by_true_class.csv",
        index=False,
    )


# ==================================================================================================
# 16. EXPERIMENT-WISE ERROR RATE
# ==================================================================================================

experiment_error = (
    ext
    .groupby(
        "exp_id"
    )[
        "correct_prediction"
    ]
    .agg(
        [
            "count",
            "mean",
        ]
    )
    .reset_index()
)


experiment_error[
    "accuracy_percent"
] = (
    100
    * experiment_error[
        "mean"
    ]
)


experiment_error[
    "error_percent"
] = (
    100
    *
    (
        1
        -
        experiment_error[
            "mean"
        ]
    )
)


print("\nExperiment-wise multiclass accuracy:")

print(
    experiment_error[
        [
            "exp_id",
            "count",
            "accuracy_percent",
            "error_percent",
        ]
    ]
    .round(2)
    .to_string(
        index=False
    )
)


experiment_error.to_csv(
    OUTPUT_DIR
    / "experiment_error_rates.csv",
    index=False,
)


# ==================================================================================================
# 17. CLASS-DISTRIBUTION SHIFT
# ==================================================================================================

print("\n" + "=" * 120)
print("CLASS DISTRIBUTION SHIFT")
print("=" * 120)


# ----------------------------------------------------------------------------------
# External GT distribution
# ----------------------------------------------------------------------------------

external_class_distribution = (
    ext[
        "true_class"
    ]
    .value_counts(
        normalize=True
    )
    .reindex(
        CLASS_ORDER,
        fill_value=0,
    )
    * 100
)


print("\nExternal Exp. 12-14 GT distribution (%):")

print(
    external_class_distribution
    .round(2)
)


# ----------------------------------------------------------------------------------
# Development target labels
#
# Try common possibilities.
# ----------------------------------------------------------------------------------

development_target_column = None


for candidate in [
    "target_id",
    "defect_class",
    "true_class",
    "class_id",
]:

    if candidate in dev.columns:

        development_target_column = candidate
        break


if development_target_column is not None:

    print(
        "\nDevelopment target column:",
        development_target_column
    )


    if development_target_column == "target_id":

        dev_class_mapping = {
            0: "Good",
            1: "Burr",
            2: "Flash-burr",
            3: "Surface-groove/void",
        }


        development_classes = pd.to_numeric(
            dev[
                development_target_column
            ],
            errors="coerce",
        ).map(
            dev_class_mapping
        )


    else:

        development_classes = (
            dev[
                development_target_column
            ]
            .apply(
                normalize_class
            )
        )


    development_class_distribution = (
        development_classes
        .value_counts(
            normalize=True
        )
        .reindex(
            CLASS_ORDER,
            fill_value=0,
        )
        * 100
    )


    print("\nDevelopment Exp. 1-11 distribution (%):")

    print(
        development_class_distribution
        .round(2)
    )


    class_shift_table = pd.DataFrame(
        {
            "class":
                CLASS_ORDER,

            "development_percent":
                development_class_distribution.values,

            "external_percent":
                external_class_distribution.values,
        }
    )


    class_shift_table[
        "difference_percentage_points"
    ] = (
        class_shift_table[
            "external_percent"
        ]
        -
        class_shift_table[
            "development_percent"
        ]
    )


    class_shift_table.to_csv(
        OUTPUT_DIR
        / "class_distribution_shift.csv",
        index=False,
    )


# ==================================================================================================
# 18. TEMPORAL GT TRANSITION ANALYSIS
# ==================================================================================================

print("\n" + "=" * 120)
print("TEMPORAL TRANSITION ERROR ANALYSIS")
print("=" * 120)


ext = ext.sort_values(
    [
        "exp_id",
        "target_frame",
    ]
).reset_index(
    drop=True
)


ext[
    "previous_true_class"
] = (
    ext
    .groupby(
        "exp_id"
    )[
        "true_class"
    ]
    .shift(1)
)


ext[
    "is_class_transition"
] = (
    ext[
        "true_class"
    ]
    !=
    ext[
        "previous_true_class"
    ]
)


# First frame of each experiment should not count as transition
first_rows = (
    ext
    .groupby(
        "exp_id"
    )
    .head(1)
    .index
)


ext.loc[
    first_rows,
    "is_class_transition",
] = False


transition_summary = (
    ext
    .groupby(
        "is_class_transition"
    )[
        "correct_prediction"
    ]
    .agg(
        [
            "count",
            "mean",
        ]
    )
    .reset_index()
)


transition_summary[
    "accuracy_percent"
] = (
    transition_summary[
        "mean"
    ]
    * 100
)


print(
    transition_summary[
        [
            "is_class_transition",
            "count",
            "accuracy_percent",
        ]
    ]
    .round(2)
    .to_string(
        index=False
    )
)


transition_summary.to_csv(
    OUTPUT_DIR
    / "temporal_transition_accuracy.csv",
    index=False,
)


# ==================================================================================================
# 19. HIGH-CONFIDENCE ERRORS
# ==================================================================================================

if "max_probability" in ext.columns:

    high_conf_errors = (
        ext[
            ~ext[
                "correct_prediction"
            ]
        ]
        .sort_values(
            "max_probability",
            ascending=False,
        )
    )


    columns_to_save = [
        col
        for col in [
            "exp_id",
            EXT_FRAME,
            "target_frame",
            "true_class",
            "pred_class",
            "max_probability",
        ]
        +
        FINAL_FEATURES
        if col in high_conf_errors.columns
    ]


    high_conf_errors[
        columns_to_save
    ].to_csv(
        OUTPUT_DIR
        / "high_confidence_errors.csv",
        index=False,
    )


    print(
        "\nTop 20 highest-confidence errors:"
    )


    print(
        high_conf_errors[
            [
                "exp_id",
                "target_frame",
                "true_class",
                "pred_class",
                "max_probability",
            ]
        ]
        .head(20)
        .round(4)
        .to_string(
            index=False
        )
    )


# ==================================================================================================
# 20. CORRECT VS INCORRECT FEATURE BEHAVIOR
# ==================================================================================================

error_feature_rows = []


for feature in FINAL_FEATURES:

    correct_values = pd.to_numeric(
        ext.loc[
            ext[
                "correct_prediction"
            ],
            feature,
        ],
        errors="coerce",
    )


    incorrect_values = pd.to_numeric(
        ext.loc[
            ~ext[
                "correct_prediction"
            ],
            feature,
        ],
        errors="coerce",
    )


    error_feature_rows.append(
        {
            "feature":
                feature,

            "correct_mean":
                correct_values.mean(),

            "incorrect_mean":
                incorrect_values.mean(),

            "correct_sd":
                correct_values.std(
                    ddof=1
                ),

            "incorrect_sd":
                incorrect_values.std(
                    ddof=1
                ),
        }
    )


error_feature_df = pd.DataFrame(
    error_feature_rows
)


error_feature_df.to_csv(
    OUTPUT_DIR
    / "correct_vs_incorrect_feature_statistics.csv",
    index=False,
)


# ==================================================================================================
# 21. PLOT — STANDARDIZED FEATURE SHIFT
# ==================================================================================================

plot_df = comparison.sort_values(
    "absolute_standardized_shift",
    ascending=True,
)


fig, ax = plt.subplots(
    figsize=(
        10,
        8,
    )
)


ax.barh(
    plot_df[
        "feature"
    ],
    plot_df[
        "standardized_mean_shift"
    ],
)


ax.axvline(
    0,
    linewidth=1,
)


ax.set_xlabel(
    "Standardized external mean shift relative to Exp. 1–11 SD"
)

ax.set_ylabel(
    "Feature"
)

ax.set_title(
    "Feature distribution shift: Exp. 1–11 vs independent Exp. 12–14"
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "feature_standardized_mean_shift.png",
    dpi=300,
    bbox_inches="tight",
)


plt.close(fig)


# ==================================================================================================
# 22. PLOT — OUTSIDE DEVELOPMENT RANGE
# ==================================================================================================

plot_range = comparison.sort_values(
    "external_outside_dev_minmax_percent",
    ascending=True,
)


fig, ax = plt.subplots(
    figsize=(
        10,
        8,
    )
)


ax.barh(
    plot_range[
        "feature"
    ],
    plot_range[
        "external_outside_dev_minmax_percent"
    ],
)


ax.set_xlabel(
    "External observations outside Exp. 1–11 min–max range (%)"
)

ax.set_ylabel(
    "Feature"
)

ax.set_title(
    "Out-of-development-range feature observations"
)


fig.tight_layout()


fig.savefig(
    OUTPUT_DIR
    / "feature_outside_development_range.png",
    dpi=300,
    bbox_inches="tight",
)


plt.close(fig)


# ==================================================================================================
# 23. PLOT — CLASS DISTRIBUTION SHIFT
# ==================================================================================================

if development_target_column is not None:

    x = np.arange(
        len(
            CLASS_ORDER
        )
    )


    width = 0.36


    fig, ax = plt.subplots(
        figsize=(
            9,
            6,
        )
    )


    ax.bar(
        x - width / 2,
        class_shift_table[
            "development_percent"
        ],
        width,
        label="Exp. 1–11",
    )


    ax.bar(
        x + width / 2,
        class_shift_table[
            "external_percent"
        ],
        width,
        label="Exp. 12–14",
    )


    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        CLASS_ORDER,
        rotation=20,
        ha="right",
    )


    ax.set_ylabel(
        "Class distribution (%)"
    )


    ax.set_title(
        "Target-class distribution shift"
    )


    ax.legend()


    fig.tight_layout()


    fig.savefig(
        OUTPUT_DIR
        / "class_distribution_shift.png",
        dpi=300,
        bbox_inches="tight",
    )


    plt.close(fig)


# ==================================================================================================
# 24. PLOT — CONFIDENCE CORRECT VS INCORRECT
# ==================================================================================================

if "max_probability" in ext.columns:

    correct_conf = ext.loc[
        ext[
            "correct_prediction"
        ],
        "max_probability",
    ]


    incorrect_conf = ext.loc[
        ~ext[
            "correct_prediction"
        ],
        "max_probability",
    ]


    fig, ax = plt.subplots(
        figsize=(
            8,
            6,
        )
    )


    ax.boxplot(
        [
            correct_conf.dropna(),
            incorrect_conf.dropna(),
        ],
        labels=[
            "Correct",
            "Incorrect",
        ],
    )


    ax.set_ylabel(
        "Maximum predicted probability"
    )


    ax.set_title(
        "Prediction confidence for correct and incorrect zero-shot predictions"
    )


    fig.tight_layout()


    fig.savefig(
        OUTPUT_DIR
        / "confidence_correct_vs_incorrect.png",
        dpi=300,
        bbox_inches="tight",
    )


    plt.close(fig)


# ==================================================================================================
# 25. MANUSCRIPT-READY DIAGNOSTIC SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("MANUSCRIPT-READY GENERALIZATION DIAGNOSTIC")
print("=" * 120)


largest_shifts = comparison.head(
    10
)


print(
    "\nTen largest pooled feature shifts:"
)


print(
    largest_shifts[
        [
            "feature",
            "standardized_mean_shift",
            "shift_magnitude",
            "external_outside_dev_minmax_percent",
        ]
    ]
    .round(3)
    .to_string(
        index=False
    )
)


print(
    "\nFeature-group shift summary:"
)


print(
    group_df
    .round(3)
    .to_string(
        index=False
    )
)


if "max_probability" in ext.columns:

    mean_correct_conf = ext.loc[
        ext[
            "correct_prediction"
        ],
        "max_probability",
    ].mean()


    mean_error_conf = ext.loc[
        ~ext[
            "correct_prediction"
        ],
        "max_probability",
    ].mean()


    print(
        f"\nMean confidence — correct predictions   : "
        f"{mean_correct_conf:.3f}"
    )

    print(
        f"Mean confidence — incorrect predictions : "
        f"{mean_error_conf:.3f}"
    )


print("\nResults saved to:")
print(
    OUTPUT_DIR
)


print("\nKey diagnostic outputs:")

for name in [
    "development_vs_external_feature_shift.csv",
    "per_experiment_feature_shift.csv",
    "feature_group_shift_summary.csv",
    "class_distribution_shift.csv",
    "experiment_error_rates.csv",
    "confidence_correct_vs_incorrect.csv",
    "confidence_by_true_class.csv",
    "temporal_transition_accuracy.csv",
    "high_confidence_errors.csv",
    "correct_vs_incorrect_feature_statistics.csv",
    "feature_standardized_mean_shift.png",
    "feature_outside_development_range.png",
    "class_distribution_shift.png",
    "confidence_correct_vs_incorrect.png",
]:

    print(
        " ",
        name
    )


print("\n" + "=" * 120)
print("DIAGNOSTIC COMPLETE")
print("=" * 120)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# OUTPUT DIRECTORY
# ============================================================

output_dir = Path.cwd() / "external_test" / "FINAL_H10_TARGET_CLASS_SHIFT"

output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# DATA
# ============================================================

classes = [
    "Good",
    "Burr",
    "Flash-burr",
    "Surface-groove/void",
]

development = [
    18.52,
    37.14,
    21.38,
    22.97,
]

external = [
    41.85,
    18.97,
    9.71,
    29.46,
]

# ============================================================
# FIGURE
# ============================================================

x = np.arange(len(classes))
width = 0.36

fig, ax = plt.subplots(figsize=(8.5, 5.5))

bars1 = ax.bar(
    x - width / 2,
    development,
    width,
    label="Exp. 1–11 development"
)

bars2 = ax.bar(
    x + width / 2,
    external,
    width,
    label="Exp. 12–14 independent"
)

# ============================================================
# VALUE LABELS
# ============================================================

for bar in bars1:
    height = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.8,
        f"{height:.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

for bar in bars2:
    height = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.8,
        f"{height:.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# ============================================================
# FORMATTING
# ============================================================

ax.set_ylabel(
    "H = 10 target prevalence (%)",
    fontsize=11
)

ax.set_xlabel(
    "Future weld-quality class",
    fontsize=11
)

ax.set_xticks(x)

ax.set_xticklabels(
    classes,
    fontsize=10
)

ax.set_ylim(
    0,
    48
)

ax.legend(
    frameon=False,
    fontsize=10
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.35
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

# ============================================================
# SAVE
# ============================================================

png_path = (
    output_dir
    / "Fig13_H10_Target_Class_Distribution_Shift.png"
)

pdf_path = (
    output_dir
    / "Fig13_H10_Target_Class_Distribution_Shift.pdf"
)

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(png_path)
print(pdf_path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

eval_dir = Path.cwd() / "external_test" / "FINAL_H10_ZERO_SHOT_COCO_EVALUATION_CORRECTED"

raw_cm_file = (
    eval_dir
    / "pooled_multiclass_confusion_matrix_RAW.csv"
)

output_dir = eval_dir

# ============================================================
# CLASS ORDER
# ============================================================

classes = [
    "Good",
    "Burr",
    "Flash-burr",
    "Surface-groove/void",
]

# ============================================================
# LOAD RAW CONFUSION MATRIX
# ============================================================

if not raw_cm_file.exists():
    raise FileNotFoundError(
        f"Confusion matrix file not found:\n{raw_cm_file}"
    )

cm_df = pd.read_csv(
    raw_cm_file,
    index_col=0
)

cm = cm_df.to_numpy(dtype=float)

print("Raw confusion matrix:")
print(cm)

# ============================================================
# ROW NORMALIZATION
# ============================================================

row_sums = cm.sum(
    axis=1,
    keepdims=True
)

cm_normalized = np.divide(
    cm,
    row_sums,
    out=np.zeros_like(
        cm,
        dtype=float
    ),
    where=row_sums != 0
) * 100

print("\nRow-normalized confusion matrix (%):")
print(
    np.round(
        cm_normalized,
        2
    )
)

# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.3, 6.2)
)

image = ax.imshow(
    cm_normalized
)

fig.colorbar(
    image,
    ax=ax,
    label="Row-normalized percentage (%)"
)

# ============================================================
# AXES
# ============================================================

ax.set_xticks(
    np.arange(
        len(classes)
    )
)

ax.set_yticks(
    np.arange(
        len(classes)
    )
)

ax.set_xticklabels(
    classes,
    rotation=25,
    ha="right"
)

ax.set_yticklabels(
    classes
)

ax.set_xlabel(
    "Predicted future weld-quality class",
    fontsize=11
)

ax.set_ylabel(
    "True future weld-quality class",
    fontsize=11
)

# ============================================================
# CELL VALUES
# ============================================================

threshold = (
    cm_normalized.max()
    / 2
)

for i in range(
    cm_normalized.shape[0]
):

    for j in range(
        cm_normalized.shape[1]
    ):

        value = cm_normalized[
            i,
            j
        ]

        raw_value = int(
            cm[
                i,
                j
            ]
        )

        text = (
            f"{value:.1f}%\n"
            f"({raw_value})"
        )

        ax.text(
            j,
            i,
            text,
            ha="center",
            va="center",
            fontsize=9
        )

# ============================================================
# FORMATTING
# ============================================================

ax.set_title(
    "Independent H = 10 zero-shot multiclass prediction",
    fontsize=11
)

fig.tight_layout()

# ============================================================
# SAVE
# ============================================================

png_path = (
    output_dir
    / "Fig14_H10_ZeroShot_Normalized_Confusion_Matrix.png"
)

pdf_path = (
    output_dir
    / "Fig14_H10_ZeroShot_Normalized_Confusion_Matrix.pdf"
)

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(png_path)
print(pdf_path)